
# Does `complex_baseband` mode's noise generator use the right PSD convention?

**Question this notebook answers:** `si_qfi`'s noise pipeline feeds the *same* numerical
one-sided power spectral density `S_v` (from `schematic.noise.get_noise_source_psd()`,
e.g. SI's own Johnson-noise formula `sqrt(4kTR)`, or a user override) into **both**
`noise/realization.py`'s `generate_baseband_noise()` (used by `complex_baseband` mode)
and `generate_rf_noise()` (used by `real_axis` mode). For the *same* physical noise
source, do the two modes end up representing the *same amount* of noise power?

**Short answer, to be verified below:** No. `generate_baseband_noise()` appears to be
missing a factor of 4 in variance (2 in amplitude) relative to the standard, textbook
relationship between a real bandpass process's one-sided PSD and its complex-envelope
representation — the one `real_axis` mode + `quantum.demodulate()` already implements
correctly.

This notebook builds that conclusion up in small, independently-checkable steps, each
with a numerical assertion, rather than asking you to trust a single derivation. Where
an earlier hand-derivation (in chat, not here) went wrong, this notebook works out the
correct relationship numerically instead of re-deriving it by hand a second time — the
goal is a result you can verify by *running this*, not by re-reading algebra.


In [1]:

import numpy as np
from scipy.signal import butter, filtfilt
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rng_seed = 0



## Part 1 — Baseline: what does each generator advertise, in isolation?

Before comparing the two modes to each other, first pin down what each generator
*already* claims to do on its own (this matches `tests/test_noise.py`'s own unit
tests — reproduced here standalone so this notebook doesn't have to trust that file).

Convention used throughout: `S_v` is a **one-sided** PSD in V²/Hz (matches the
`single_sided_psd_v2_per_hz` naming in `noise/psd.py`, and SI's own Johnson formula
`sqrt(4·k_B·T·R)`, itself a standard one-sided physical PSD).


In [2]:

import sys
sys.path.insert(0, r"c:\Users\Daniel\Documents\GitHub\si_qfi")
from si_qfi.noise.realization import generate_baseband_noise, generate_rf_noise

N = 400_000
fs = 40e9
S_v = 1e-15  # V^2/Hz, one-sided

# --- generate_baseband_noise: claims Var = S_v * fs ---
rng = np.random.default_rng(rng_seed)
v_bb = generate_baseband_noise(N, fs, np.full(N, S_v), rng=rng)
var_bb = np.var(v_bb)
print(f"generate_baseband_noise: measured Var = {var_bb:.4e}, S_v*fs = {S_v*fs:.4e}, ratio = {var_bb/(S_v*fs):.4f}")
assert abs(var_bb/(S_v*fs) - 1) < 0.05

# --- generate_rf_noise: claims Var = S_v * fs (real-valued) ---
rng = np.random.default_rng(rng_seed)
v_rf = generate_rf_noise(N, fs, np.full(N//2+1, S_v), rng=rng)
var_rf = np.var(v_rf)
print(f"generate_rf_noise:       measured Var = {var_rf:.4e}, S_v*fs = {S_v*fs:.4e}, ratio = {var_rf/(S_v*fs):.4f}")
assert abs(var_rf/(S_v*fs) - 1) < 0.05
print("\nBoth generators individually match their own advertised Var = S_v * fs convention.")
print("So this is NOT a bug in either generator's own internal self-consistency --")
print("the question is whether that SHARED convention is the physically right one")
print("once you compare what happens when the SAME S_v is meant to represent the")
print("SAME physical noise source, viewed through each mode's own representation.")


generate_baseband_noise: measured Var = 4.0035e-05, S_v*fs = 4.0000e-05, ratio = 1.0009
generate_rf_noise:       measured Var = 4.0111e-05, S_v*fs = 4.0000e-05, ratio = 1.0028

Both generators individually match their own advertised Var = S_v * fs convention.
So this is NOT a bug in either generator's own internal self-consistency --
the question is whether that SHARED convention is the physically right one
once you compare what happens when the SAME S_v is meant to represent the
SAME physical noise source, viewed through each mode's own representation.



## Part 2 — The real question: same `S_v`, does `real_axis`+demodulate agree with `complex_baseband`?

This is the actual comparison that matters for `si_qfi`: `schematic.noise.get_noise_source_psd()`
hands the identical `S_v` array to both code paths. `real_axis` mode draws real noise at the
schematic's native rate and (for anything downstream that needs a rotating-frame
Hamiltonian, e.g. `gate_fidelity()`) demodulates it via `quantum.demodulate()`, which
applies a "×2" amplitude correction — see its own docstring, `"×2 to correct for
single-sideband"`.

So: draw real noise with a KNOWN, controlled one-sided PSD `S_v` at a high native rate,
demodulate it down to a complex baseband-equivalent bandwidth `B`, and compare its
variance against what `generate_baseband_noise()` gives directly for the same `S_v` at
a matching bandwidth (`fs_bb = 2B`).


In [3]:

from si_qfi.quantum.hamiltonian import demodulate

N = 1_000_000
fs_native = 40e9
carrier_hz = 5e9
B = 2e9          # demodulation LPF cutoff -- matches a complex_baseband fs_bb = 2*B
S_v = 1e-15

rng = np.random.default_rng(1)
v_rf = generate_rf_noise(N, fs_native, np.full(N//2+1, S_v), rng=rng)

t = np.arange(N) / fs_native
I, Q = demodulate(v_rf, t, carrier_hz, lpf_cutoff_hz=B)
v_demod = I + 1j*Q
edge = N // 10   # drop filter-edge transients
var_demod = np.var(v_demod[edge:-edge])

fs_bb = 2 * B
rng2 = np.random.default_rng(2)
v_bb_direct = generate_baseband_noise(N, fs_bb, np.full(N, S_v), rng=rng2)
var_direct = np.var(v_bb_direct)

print(f"real_axis noise, demodulated to baseband-equivalent:  Var = {var_demod:.4e}")
print(f"complex_baseband noise, generated directly (same S_v): Var = {var_direct:.4e}")
print(f"ratio (demodulated / direct)                         = {var_demod/var_direct:.3f}")
print()
print("If the two modes agreed, this ratio would be ~1.0. It is not.")


real_axis noise, demodulated to baseband-equivalent:  Var = 1.5150e-05
complex_baseband noise, generated directly (same S_v): Var = 3.9986e-06
ratio (demodulated / direct)                         = 3.789

If the two modes agreed, this ratio would be ~1.0. It is not.



## Part 3 — First-principles derivation of what the ratio *should* be

Setup: a real signal `v_rf(t)` with **one-sided** PSD `S_v` (flat/white), so
`Var(v_rf) = ∫₀^∞ S_v df` restricted to whatever band it's actually drawn over.

Demodulation, matching `quantum.demodulate()`'s own implementation exactly:

```
mix(t)  = 2 · v_rf(t) · exp(-i·2π·f_c·t)
v_bb(t) = LPF_B[ mix(t) ]           # complex, LPF cutoff B
```

**Step 1 — what survives the mix + LPF.** In the frequency domain, multiplying by
`exp(-i2πf_c t)` shifts the spectrum: `Mix(f) = 2·V_rf(f + f_c)`. Low-pass filtering to
`|f| < B` keeps only `V_rf` evaluated over the *original* frequency window
`[f_c - B, f_c + B]` — a one-sided slice of width `2B`. Nothing else in `v_rf`'s
spectrum (near DC, near `2f_c`, etc.) survives.

**Step 2 — power in that slice.** Because `S_v` is one-sided and flat, the power
contained in that `2B`-wide slice is exactly

```
P_slice = S_v · (2B)
```

**Step 3 — effect of the mix+LPF on power.** Mixing by a unit-modulus complex
exponential is power-preserving (pointwise `|mix(t)|` before the ×2 equals `2|v_rf(t)|`,
and low-pass filtering only removes out-of-band content, it doesn't rescale what's
kept). So immediately after mixing+filtering, *before* accounting for the explicit ×2
already folded into `mix(t)` above, the surviving content still carries `P_slice`
worth of power — but it entered as `2·v_rf`, not `v_rf`, so the power is scaled by the
demodulator's own `2²=4`:

```
Var(v_bb) = 4 · P_slice = 4 · S_v · (2B) = 8 · S_v · B
```

Compare this against `generate_baseband_noise()`'s own convention, using
`fs_bb = 2B` (so its bandwidth matches the demodulator's `B`-wide-each-side LPF):

```
generate_baseband_noise's Var = S_v · fs_bb = S_v · 2B
```

**Predicted ratio:** `(8·S_v·B) / (S_v·2B) = 4`.

This matches Part 2's *direction* (baseband is too small) — now check the actual
number.


In [4]:

predicted_ratio = 4.0
measured_ratio = var_demod / var_direct
print(f"predicted ratio (from the derivation above): {predicted_ratio:.3f}")
print(f"measured ratio (Part 2):                     {measured_ratio:.3f}")
print(f"agreement: {measured_ratio/predicted_ratio:.1%} of predicted")
assert abs(measured_ratio/predicted_ratio - 1) < 0.15, "derivation does not match measurement within 15%"
print("\nDerivation confirmed within expected statistical/filter-rolloff tolerance.")


predicted ratio (from the derivation above): 4.000
measured ratio (Part 2):                     3.789
agreement: 94.7% of predicted

Derivation confirmed within expected statistical/filter-rolloff tolerance.



## Part 4 — Fully independent numerical check (no `si_qfi` code at all)

Part 2/3 relies on `si_qfi`'s own `generate_rf_noise()` and `demodulate()`. To rule out
the possibility that a bug in *either* of those functions is what's producing the
factor of 4 (rather than it being a real, expected property of demodulating broadband
noise), this section reimplements both from scratch using nothing but raw
`numpy`/`scipy.signal` — completely independent of anything in `si_qfi`.


In [5]:

N = 2_000_000
fs = 40e9
S_v = 1e-15
carrier = 5e9
B = 2e9

# --- independent white-noise generator with an EXACTLY controlled one-sided PSD ---
rng = np.random.default_rng(10)
n_half = N // 2 + 1
amp = np.sqrt(S_v * fs * N) / np.sqrt(2)     # solved directly from the Parseval relation
                                              # Var = (1/N) * mean|FFT|^2 for a flat one-sided PSD
Xk = amp * (rng.standard_normal(n_half) + 1j * rng.standard_normal(n_half))
Xk[0] = Xk[0].real * np.sqrt(2)
if N % 2 == 0:
    Xk[-1] = Xk[-1].real * np.sqrt(2)
v_rf_indep = np.fft.irfft(Xk, n=N)

var_check = np.var(v_rf_indep)
print(f"independent white noise: Var = {var_check:.4e}, target S_v*fs = {S_v*fs:.4e}, "
      f"ratio = {var_check/(S_v*fs):.4f}")
assert abs(var_check/(S_v*fs) - 1) < 0.05

# --- independent demodulator: standard I/Q, x = LPF[2·v·cos(wt)], y = LPF[-2·v·sin(wt)] ---
t = np.arange(N) / fs
b, a = butter(N=8, Wn=B / (fs / 2), btype="low")
mix_indep = 2.0 * v_rf_indep * np.exp(-1j * 2 * np.pi * carrier * t)
x_indep = filtfilt(b, a, mix_indep.real)
y_indep = filtfilt(b, a, mix_indep.imag)

edge = N // 10
var_env_indep = np.var(x_indep[edge:-edge]) + np.var(y_indep[edge:-edge])

print(f"\nindependent demodulated envelope: Var = {var_env_indep:.4e}")
print(f"predicted 8*S_v*B                 = {8*S_v*B:.4e}  (ratio {var_env_indep/(8*S_v*B):.3f})")
print(f"generate_baseband_noise convention = {S_v*2*B:.4e}  (ratio {var_env_indep/(S_v*2*B):.3f})")
assert abs(var_env_indep/(8*S_v*B) - 1) < 0.15
print("\nFully independent implementation confirms the same ~4x factor. Not a bug")
print("localized to si_qfi's own generate_rf_noise() or demodulate() -- it's a real")
print("property of demodulating broadband noise down to a narrow baseband slice.")


independent white noise: Var = 3.9997e-05, target S_v*fs = 4.0000e-05, ratio = 0.9999



independent demodulated envelope: Var = 1.5117e-05
predicted 8*S_v*B                 = 1.6000e-05  (ratio 0.945)
generate_baseband_noise convention = 4.0000e-06  (ratio 3.779)

Fully independent implementation confirms the same ~4x factor. Not a bug
localized to si_qfi's own generate_rf_noise() or demodulate() -- it's a real
property of demodulating broadband noise down to a narrow baseband slice.



## Part 5 — Sanity check: does `demodulate()` correctly round-trip a *genuinely
narrowband* signal?

Parts 2–4 all involve demodulating *broadband* noise (spanning far more bandwidth than
the LPF keeps). As a sanity check on the demodulation machinery itself (separate from
the "how much power survives filtering broadband content" question above), this part
checks the simpler case: if a signal is *already* narrowband before demodulating, does
`demodulate()`'s "×2" exactly recover it, with no factor-of-4 surprise? This should
hold for both a deterministic pulse and band-limited noise, and confirms the ×2 itself
is implemented correctly — the factor of 4 above is a property of *what's being
demodulated* (broadband vs. narrowband), not a flaw in `demodulate()`.


In [6]:

# --- 5a: deterministic narrowband envelope ---
N = 20000
fs_native = 40e9
carrier_hz = 5e9
fs_bb = 4e9

t_bb = np.arange(N // 10) / fs_bb
v_bb_known = np.exp(-((t_bb - t_bb[-1]/2)**2) / (2*(t_bb[-1]/6)**2)).astype(complex)

upsample = int(round(fs_native / fs_bb))
v_bb_up = np.repeat(v_bb_known, upsample)
t_up = np.arange(len(v_bb_up)) / fs_native
v_rf_det = np.real(v_bb_up * np.exp(1j * 2*np.pi*carrier_hz*t_up))

I, Q = demodulate(v_rf_det, t_up, carrier_hz, lpf_cutoff_hz=fs_bb/2)
v_recovered_det = I + 1j*Q

# compare peak magnitude (a narrowband deterministic pulse's peak should be recovered almost exactly)
print(f"deterministic pulse: original peak |v_bb| = {np.max(np.abs(v_bb_known)):.4f}, "
      f"recovered peak |v_bb| = {np.max(np.abs(v_recovered_det)):.4f}")


deterministic pulse: original peak |v_bb| = 1.0000, recovered peak |v_bb| = 1.0000


In [7]:

# --- 5b: band-limited (narrowband) NOISE, modulated up and back down ---
# Unlike Part 2-4, v_bb here is deliberately built with ZERO power outside its own
# +/-B window BEFORE modulating up -- so the resulting v_rf is genuinely narrowband,
# not broadband white noise.
N_up = 2_000_000
fs_native = 40e9
fs_bb = 4e9
B = fs_bb / 2
S_v_input = 1e-15
carrier_hz = 5e9

freqs_full = np.fft.fftfreq(N_up, d=1/fs_native)
psd_bandlimited = np.where(np.abs(freqs_full) < B, S_v_input, 0.0)

rng = np.random.default_rng(20)
v_bb_narrowband = generate_baseband_noise(N_up, fs_native, psd_bandlimited, rng=rng)
var_bb_start = np.var(v_bb_narrowband)
print(f"starting narrowband baseband noise: Var = {var_bb_start:.4e} "
      f"(target S_v_input*fs_bb = {S_v_input*fs_bb:.4e})")

t_up = np.arange(N_up) / fs_native
v_rf_noise = np.real(v_bb_narrowband * np.exp(1j*2*np.pi*carrier_hz*t_up))
var_rf_modulated = np.var(v_rf_noise)
print(f"modulated-up real signal: Var = {var_rf_modulated:.4e} "
      f"(standard bandpass relation predicts Var(v_bb)/2 = {var_bb_start/2:.4e})")

I, Q = demodulate(v_rf_noise, t_up, carrier_hz, lpf_cutoff_hz=B)
v_recovered = I + 1j*Q
edge = N_up // 10
var_recovered = np.var(v_recovered[edge:-edge])
print(f"\nrecovered (round-tripped) baseband noise: Var = {var_recovered:.4e}")
print(f"ratio recovered/original: {var_recovered/var_bb_start:.3f}  (expect ~1.0 for a clean round trip)")


starting narrowband baseband noise: Var = 3.9982e-06 (target S_v_input*fs_bb = 4.0000e-06)
modulated-up real signal: Var = 1.9991e-06 (standard bandpass relation predicts Var(v_bb)/2 = 1.9991e-06)



recovered (round-tripped) baseband noise: Var = 3.7302e-06
ratio recovered/original: 0.933  (expect ~1.0 for a clean round trip)



### Why 5b is *not* the same comparison as Part 2, and doesn't contradict it

Part 5b starts from noise that is **already confined** to `[f_c-B, f_c+B]` (and its
mirror) before demodulating — genuinely narrowband, like the deterministic case in 5a.
Demodulating recovers close to the original variance: no factor of 4 anywhere, because
there's no broadband content being *thrown away* by the filter in the first place —
everything the LPF keeps is everything there was.

Part 2–4 draw noise that is white across the **entire native bandwidth** (`generate_rf_noise`,
by construction, and by design — it's meant to represent the qubit's actual physical
noise environment, not something pre-shaped to match the drive pulse's own narrow
bandwidth). Demodulating that noise means the LPF is discarding the vast majority of
the noise's total power (everything outside `[f_c-B, f_c+B]`) and keeping *only* a
`2B`-wide slice. The factor of 4 comes from the ×2 demodulation gain applied to *that
slice specifically* — a real, physically meaningful quantity (the noise power that
actually lands in-band at the qubit), not an artifact of the round-trip machinery.

**This is exactly why `generate_baseband_noise()` needs the same factor**: physically,
a Johnson noise source's `S_v` describes noise spread across a wide bandwidth (in this
codebase's own numbers, `EndFrequency` ~ 20 GHz), the same way `generate_rf_noise()`
already treats it. `generate_baseband_noise()` currently treats the *same* `S_v` value
as if it already represented ONLY the narrow baseband slice's own local density,
missing the "how much of the wideband source actually lands in this slice" factor
that `real_axis` + `demodulate()` correctly compute.



## Conclusion

1. **`generate_rf_noise()`** and **`generate_baseband_noise()`** are each internally
   self-consistent with their own advertised `Var = S_v · fs` convention (Part 1).
2. **They disagree with each other** for the same physical `S_v`: demodulating
   `real_axis` noise gives ~4x the variance that `generate_baseband_noise()` computes
   directly for the same `S_v` and matching bandwidth (Part 2).
3. **A first-principles derivation predicts exactly this factor of 4** (Part 3),
   confirmed by a from-scratch, `si_qfi`-independent reimplementation (Part 4).
4. **The factor is specific to demodulating *broadband* noise**, not a flaw in
   `demodulate()`'s own ×2 correction — a genuinely narrowband source round-trips
   correctly with no such factor (Part 5).
5. Since `generate_rf_noise()` correctly represents a physical, wideband one-sided
   `S_v` (matching how SI's own Johnson-noise formula, and this codebase's own
   `single_sided_psd_v2_per_hz` override, are meant to be interpreted), the fix
   belongs in **`generate_baseband_noise()`** — it should produce `Var = 4 · S_v · fs`,
   not `S_v · fs`, to represent the same physical source consistently with `real_axis`
   mode.

**Recommended fix** (not applied in this notebook): in `noise/realization.py`,
`generate_baseband_noise()`'s per-bin amplitude `amp = np.sqrt(psd_two_sided * df)`
would become `amp = np.sqrt(psd_two_sided * df) * 2.0`, doubling amplitude / quadrupling
variance. This changes every existing `complex_baseband`-mode noise result in this
codebase by exactly 4x (all of it understated) and needs downstream re-verification
(the noise-density-sweep demo, the engine-level noise RMS tests) once applied.
